# HIG₄,₈ geométrico aplicado a conectomas funcionales reales
**Notebook científico derivado de `HIGM48_editor_20x20_Colab.ipynb` (motor de construcción conservado).**

**Objeto del experimento:** matrices simétricas de correlación entre ROI de fMRI, con atlas/orden idénticos entre sujetos. Este notebook **no implementa** el hipergrafo de ROI de Carlos: es exclusivamente la vía geométrica del **HIG de la imagen de correlaciones**, remitiendo al artículo HIG original para sus definiciones y propiedades.

**Secuencia:** importar matrices reales → inspección/QC → cuantización firmada → HIG₄ y HIG₈ combinatorio → métricas e incidencias → baselines RAG/topología/métrica → validación entre sujetos → exportación.

**Datos:** sube archivos `.csv`, `.txt` o `.npy` N×N, o un ZIP que los contenga. Opcionalmente carga `manifest.csv` con columnas `filename,subject,label` (y `family`, si disponible). Los ficheros pueden ser matrices Pearson *r* o Fisher-*z* (configurable); no se descargan registros personales ni se fabrican datos clínicos. Un ejemplo sintético sirve solo para comprobar el funcionamiento, no como resultado empírico.

**Precauciones:** las entradas deben compartir exactamente el mismo atlas/orden ROI; aquí no se estima conectividad dinámica a partir de un único volumen fMRI. Un registro `filename` representa una matriz ya estimada. Etiquetas derivadas de la condición experimental o diagnóstico, no del propio HIG. Para HCP y Schaefer, documentar explícitamente el atlas, el mapeo y el nivel de agregación temporal.

In [ ]:
import io, os, json, zipfile, warnings, pathlib, time
from collections import Counter, defaultdict, deque
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import sparse, ndimage
from IPython.display import display
try:
    from google.colab import files
    IN_COLAB=True
except ImportError:
    IN_COLAB=False
from sklearn.model_selection import StratifiedGroupKFold, GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
ROOT=pathlib.Path('/content/hig_connectome' if IN_COLAB else './hig_connectome')
ROOT.mkdir(parents=True,exist_ok=True)
print('Directorio:',ROOT,' Colab:',IN_COLAB)

## 1. Configuración y entrada de datos

Carga matrices individuales y un `manifest.csv` con `filename,subject,label` (preferiblemente también `family`). Si no hay manifest se permitirá explorar los conectomas, pero **no** entrenar un clasificador. Las matrices con distinta dimensión o asimetría fuerte serán rechazadas. Selecciona el tipo de valores y el tratamiento de la diagonal. El orden semántico de ROI es fijo: no ordenar ni normalizar las filas individualmente por sujeto.

In [ ]:
# AJUSTAR ANTES DEL EXPERIMENTO
INPUT_SCALE='r'  # 'r' para Pearson; 'z' para Fisher-z, se aplica tanh antes de cuantizar
THRESHOLDS=(-0.50,-0.15,0.15,0.50)
DIAGONAL_MODE='separate' # etiqueta exclusiva (constante) para la diagonal; controles: 'neutral'
EXPECTED_N=None          # establecer 200 para Schaefer-200, 259 si se usa el dataset del TFM
MAX_MATRIX_SIZE=500      # guardarraíl de recursos para Google Colab
OUT=ROOT/'uploads'; OUT.mkdir(exist_ok=True)
print('Configuración:',INPUT_SCALE,THRESHOLDS,DIAGONAL_MODE)

In [ ]:
# Ejecutar en Colab y seleccionar matrices y opcionalmente manifest.csv, o un ZIP.
if IN_COLAB:
    uploaded=files.upload()
    for name,data in uploaded.items():
        safe=pathlib.Path(name).name
        if safe.lower().endswith('.zip'):
            with zipfile.ZipFile(io.BytesIO(data)) as zz:
                for info in zz.infolist():
                    fn=pathlib.Path(info.filename).name
                    if fn and not info.is_dir() and pathlib.Path(fn).suffix.lower() in {'.csv','.txt','.npy'}:
                        (OUT/fn).write_bytes(zz.read(info))
        elif pathlib.Path(safe).suffix.lower() in {'.csv','.txt','.npy'}:
            (OUT/safe).write_bytes(data)
print('Archivos disponibles:',sorted(p.name for p in OUT.iterdir()))

In [ ]:
def load_matrix(path):
    path=pathlib.Path(path)
    if path.suffix.lower()=='.npy': c=np.load(path,allow_pickle=False)
    else:
        try: c=np.loadtxt(path, delimiter=',' if path.suffix.lower()=='.csv' else None)
        except ValueError:
            c=pd.read_csv(path,header=None).values.astype(float)
    c=np.asarray(c,dtype=float)
    if c.ndim!=2 or c.shape[0]!=c.shape[1] or c.shape[0]<3: raise ValueError(f'{path.name}: debe ser cuadrada N>=3')
    if c.shape[0]>MAX_MATRIX_SIZE: raise ValueError('Matriz demasiado grande para este prototipo')
    if EXPECTED_N is not None and len(c)!=EXPECTED_N: raise ValueError('N incompatible con el atlas esperado')
    if not np.isfinite(c).all(): raise ValueError(f'{path.name}: hay NaN o Inf (imputación prohibida en QC)')
    err=float(np.max(np.abs(c-c.T)))
    if err>1e-4: raise ValueError(f'{path.name}: no es simétrica (max error={err:.4g})')
    c=(c+c.T)/2
    if INPUT_SCALE=='z': c=np.tanh(c)
    elif INPUT_SCALE!='r': raise ValueError('INPUT_SCALE debe ser r o z')
    if np.max(np.abs(c))>1.0001: raise ValueError(f'{path.name}: fuera de [-1,1] tras transformación')
    np.fill_diagonal(c,0.0)
    return c

def labels_from_correlations(c, thresholds=THRESHOLDS):
    q=np.digitize(c, thresholds).astype(np.int32)
    if DIAGONAL_MODE=='separate': np.fill_diagonal(q, len(thresholds)+1)
    elif DIAGONAL_MODE=='neutral': np.fill_diagonal(q,2)
    else: raise ValueError('modo diagonal inválido')
    return q

paths=sorted(p for p in OUT.iterdir() if p.suffix.lower() in {'.csv','.npy','.txt'} and p.name!='manifest.csv')
print(f'Se encontraron {len(paths)} matrices. Si hay cero, ejecuta la celda de carga.')
if paths:
    first=load_matrix(paths[0]); print('Ejemplo:',paths[0].name,'N=',len(first),'rango fuera de diagonal:',(first[~np.eye(len(first),dtype=bool)].min(),first.max()))

## 2. Motor HIG₄ exacto reutilizado del notebook original

La construcción por continuación de fronteras, los tipos `seg`, `enc0`, `enc1`, los generadores de corte y las matrices dispersas permanecen fieles al prototipo aportado. No se redefine aquí la teoría HIG; el artículo anterior es la referencia matemática. Los perímetros se calculan desde los **bordes de la cuadrícula**; no se confunden los agujeros de la imagen con ciclos del conectoma de ROI.

In [ ]:
from collections import Counter, defaultdict, deque
import numpy as np
from scipy import sparse

# Implementación autocontenida de la construcción intrínseca HIG₄.
# No necesita archivos auxiliares cuando se abre en Colab.

def region_ids(image):
    """Four-connected equal-label components; OUT receives the final id."""
    h, w = image.shape
    ids = np.full((h, w), -1, dtype=np.int32)
    nr = 0
    for y in range(h):
        for x in range(w):
            if ids[y, x] != -1:
                continue
            label = image[y, x]
            ids[y, x] = nr
            queue = deque([(y, x)])
            while queue:
                yy, xx = queue.popleft()
                for ny, nx in ((yy-1, xx), (yy+1, xx), (yy, xx-1), (yy, xx+1)):
                    if 0 <= ny < h and 0 <= nx < w and ids[ny, nx] == -1 and image[ny, nx] == label:
                        ids[ny, nx] = nr
                        queue.append((ny, nx))
            nr += 1
    return ids, nr


def construct(image):
    ids, out = region_ids(image)
    h, w = image.shape
    edges = []                 # (first grid vertex, second, region pair, on frame)
    at = defaultdict(list)
    def add(a, b, u, v, frame):
        if u == v:
            return
        i = len(edges)
        pair = tuple(sorted((int(u), int(v))))
        edges.append((a, b, pair, frame))
        at[a].append(i)
        at[b].append(i)
    for y in range(h):
        for x in range(w+1):
            add((y,x), (y+1,x), out if x == 0 else ids[y,x-1],
                out if x == w else ids[y,x], x in (0,w))
    for y in range(h+1):
        for x in range(w):
            add((y,x), (y,x+1), out if y == 0 else ids[y-1,x],
                out if y == h else ids[y,x], y in (0,h))
    junctions = sorted(v for v, incident in at.items() if len(incident) >= 3)
    jset = set(junctions)
    jindex = {v:i for i,v in enumerate(junctions)}
    geom = []
    seen = set()
    for start in range(len(edges)):
        if start in seen:
            continue
        queue = [start]
        seen.add(start)
        group = []
        for i in queue:
            group.append(i)
            a,b,pair,_ = edges[i]
            for v in (a,b):
                if v in jset:
                    continue
                for nxt in at[v]:
                    if nxt not in seen and edges[nxt][2] == pair:
                        seen.add(nxt)
                        queue.append(nxt)
        endpoints = []
        for i in group:
            for v in edges[i][:2]:
                if v in jset:
                    endpoints.append(v)
        if len(endpoints) == 0:
            typ = "enc0"
        elif len(endpoints) == 2:
            typ = "enc1" if endpoints[0] == endpoints[1] else "seg"
        else:
            raise ValueError(f"Continuation has {len(endpoints)} endpoint occurrences")
        geom.append({"type":typ,"pair":pair,"endpoints":endpoints,"edges":group})
    cuts = []
    seen = set()
    for start in range(len(edges)):
        if start in seen:
            continue
        queue = [start]
        seen.add(start)
        group = []
        frame = False
        for i in queue:
            group.append(i)
            a,b,_,on_frame = edges[i]
            frame |= on_frame
            for v in (a,b):
                for nxt in at[v]:
                    if nxt not in seen:
                        seen.add(nxt)
                        queue.append(nxt)
        if not frame:
            cuts.append(group)
    # Pixel-side to frontier edge; supports local validation on small images.
    pixel_sides = defaultdict(list)
    for i,(a,b,pair,_) in enumerate(edges):
        if a[0] == b[0]:
            yy=a[0]; xx=a[1]
            for py in (yy-1,yy):
                if 0<=py<h:pixel_sides[(py,xx)].append(i)
        else:
            yy=a[0]; xx=a[1]
            for px in (xx-1,xx):
                if 0<=px<w:pixel_sides[(yy,px)].append(i)
    # Each frame-free frontier component corresponds to exactly one
    # bounded 8-component of the complement of exactly one 4-region.
    # Establish the carrier by matching its complete R-boundary to the cut.
    # Its uppermost horizontal edge has the carrier pixel immediately above.
    # This gives a linear-time carrier extraction without flooding the
    # complement separately for every region (which is prohibitive for WSI).
    carriers=[]
    for group in cuts:
        horizontal=[i for i in group if edges[i][0][0]==edges[i][1][0]]
        i=min(horizontal,key=lambda k:(edges[k][0][0],edges[k][0][1]))
        yy,xx=edges[i][0]
        carriers.append(int(ids[yy-1,xx]))
    holes=Counter(carriers)
    if h*w<=10000:
      cut_sets = [set(g) for g in cuts]
      carrier_matches = [[] for _ in cuts]
      verified_holes=Counter()
      for region in range(out):
        visited = np.zeros((h,w), dtype=bool)
        for y in range(h):
            for x in range(w):
                if ids[y,x] == region or visited[y,x]:
                    continue
                queue = deque([(y,x)])
                visited[y,x] = True
                pixels = set()
                frame = False
                while queue:
                    yy,xx = queue.popleft()
                    pixels.add((yy,xx))
                    frame |= yy in (0,h-1) or xx in (0,w-1)
                    for dy in (-1,0,1):
                        for dx in (-1,0,1):
                            ny,nx = yy+dy,xx+dx
                            if 0 <= ny < h and 0 <= nx < w and not visited[ny,nx] and ids[ny,nx] != region:
                                visited[ny,nx] = True
                                queue.append((ny,nx))
                if frame:
                    continue
                verified_holes[region] += 1
                boundary = set()
                for py,px in pixels:
                    for i in pixel_sides[(py,px)]:
                        if region in edges[i][2]:boundary.add(i)
                # C can include frontiers between regions *inside* the hole;
                # only the R--H boundary must lie in that component.
                matches = [k for k,g in enumerate(cut_sets) if boundary and boundary <= g]
                if len(matches) != 1:
                    raise ValueError(f"Cut-hole correspondence failed at region {region}: {len(matches)} matches")
                carrier_matches[matches[0]].append(region)
      if any(m != [carriers[k]] for k,m in enumerate(carrier_matches)) or holes != verified_holes:
          raise ValueError("Fast carrier disagrees with explicit 8-hole verification")
    n0 = len(junctions) + sum(g["type"] == "enc0" for g in geom)
    n1 = len(geom) + len(cuts)
    n2 = out + 1
    ii01,jj01,vv01,ii12,jj12,vv12 = [],[],[],[],[],[]
    virtual = len(junctions)
    for k,g in enumerate(geom):
        endpoints = g["endpoints"]
        if g["type"] == "enc0":
            ii01.append(virtual); jj01.append(k); vv01.append(2)
            virtual += 1
        else:
            for j,count in Counter(jindex[v] for v in endpoints).items():
                ii01.append(j); jj01.append(k); vv01.append(count)
        for region in g["pair"]:
            ii12.append(k); jj12.append(region); vv12.append(1)
    for k,region in enumerate(carriers, start=len(geom)):
        ii12.append(k); jj12.append(region); vv12.append(2)
        # Intentionally NO 0--1 incidence for cut k.
    m01 = sparse.csr_matrix((vv01,(ii01,jj01)),shape=(n0,n1),dtype=np.int32)
    m12 = sparse.csr_matrix((vv12,(ii12,jj12)),shape=(n1,n2),dtype=np.int32)
    return ids, edges, geom, cuts, carriers, holes, m01, m12

## 3. HIG₈ combinatorio, obtenido del cociente diagonal del HIG₄

Se conserva la derivación original y sus aserciones de correspondencia de agujeros. **HIG₈ es una construcción combinatoria para recuentos**: no se supone una nueva segmentación planar válida cuando hay cruces diagonales.

In [ ]:
from scipy import ndimage
def _derive_hig8(image, ids, geom, cuts, holes4, m01, m12, junctions):
    h, w = image.shape
    old_out = m12.shape[1] - 1
    parent = list(range(old_out))

    def root(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    def union(a, b):
        a, b = root(a), root(b)
        if a != b:
            parent[max(a, b)] = min(a, b)

    # q_v se lee sobre los cuatro cuadrantes completos de cada vértice interior.
    for y in range(1, h):
        for x in range(1, w):
            for (a, b) in (((y-1, x-1), (y, x)),
                           ((y-1, x), (y, x-1))):
                ra, rb = int(ids[a]), int(ids[b])
                if ra != rb and image[a] == image[b]:
                    union(ra, rb)

    roots = sorted({root(r) for r in range(old_out)})
    index = {r: i for i, r in enumerate(roots)}
    quotient = [index[root(r)] for r in range(old_out)]
    new_out = len(roots)
    quotient.append(new_out)    # OUT se transporta a OUT.
    qmap = np.asarray(quotient, dtype=int)[ids]
    junction_index = {v: i for i, v in enumerate(junctions)}
    diagonal = []
    d_by_q = Counter()
    for y in range(1, h):
        for x in range(1, w):
            nw, ne = int(qmap[y-1, x-1]), int(qmap[y-1, x])
            sw, se = int(qmap[y, x-1]), int(qmap[y, x])
            for qid, same, other_a, other_b, orientation in (
                (nw, se, ne, sw, 'NW–SE'),
                (ne, sw, nw, se, 'NE–SW')):
                if qid == same and other_a != qid and other_b != qid:
                    assert (y, x) in junction_index
                    diagonal.append({'vertex': (y, x), 'region': qid,
                                     'junction': junction_index[(y, x)],
                                     'orientation': orientation})
                    d_by_q[qid] += 1

    # Oráculo independiente para la agrupación: componentes 8 directas.
    direct = []
    for value in np.unique(image):
        labels, count = ndimage.label(image == value, structure=np.ones((3, 3), dtype=int))
        direct.extend((labels == i).tobytes() for i in range(1, count + 1))
    assert sorted(direct) == sorted((qmap == i).tobytes() for i in range(new_out))

    four_structure = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], dtype=int)
    hole_nodes = []
    regions = []
    for qid in range(new_out):
        mask = qmap == qid
        complement = np.pad(~mask, 1, constant_values=True)
        labels, count = ndimage.label(complement, structure=four_structure)
        outside = int(labels[0, 0])
        holes = [labels[1:-1, 1:-1] == i for i in range(1, count + 1) if i != outside]
        members = [r for r in range(old_out) if quotient[r] == qid]
        k = len(members)
        d = d_by_q[qid]
        h4 = sum(holes4.get(r, 0) for r in members)
        assert len(holes) == h4 + d - k + 1, (qid, k, d, h4, len(holes))
        coords = np.argwhere(mask)
        y, x = coords[0]
        regions.append({'id': qid, 'label': int(image[y, x]), 'pixels': len(coords),
                        'members': members, 'holes': len(holes), 'd': d,
                        'cycle_rank': d-k+1})
        for hole in holes:
            hole_nodes.append({'carrier': qid, 'hole_pixels': int(np.count_nonzero(hole)),
                               'first_pixel': tuple(int(c) for c in np.argwhere(hole)[0])})

    m = old_out - new_out
    d = len(diagonal)
    k4, k8 = len(cuts), len(hole_nodes)
    n0 = m01.shape[0] + d
    n1 = len(geom) + k8
    n2 = new_out + 1
    chi = n0 - n1 + n2
    assert k8-k4 == d-m
    assert chi == 2

    # Incidencias tipadas de la definición de HIG_8^{comb}.
    old_to_new = sparse.csr_matrix((np.ones(old_out+1, dtype=int),
                    (np.arange(old_out+1), quotient)), shape=(old_out+1, n2))
    retained12 = m12[:len(geom), :] @ old_to_new
    cut12 = sparse.csr_matrix((np.full(k8, 2, dtype=int),
                    (np.arange(k8), [item['carrier'] for item in hole_nodes])),
                    shape=(k8, n2))
    m12_8 = sparse.vstack([retained12, cut12], format='csr')
    old01 = m01[:, :len(geom)]
    new_rows = old01[[item['junction'] for item in diagonal], :]
    geometric01 = sparse.vstack([old01, new_rows], format='csr')
    m01_8 = sparse.hstack([geometric01, sparse.csr_matrix((n0, k8), dtype=int)], format='csr')
    assert m01_8.shape == (n0, n1) and m12_8.shape == (n1, n2)
    assert m01_8[:, len(geom):].nnz == 0
    assert all(m12_8[len(geom)+i, item['carrier']] == 2
               for i, item in enumerate(hole_nodes))

    return {'regions': regions, 'diagonal': diagonal, 'holes': hole_nodes,
            'quotient': quotient, 'm': m, 'd': d, 'k4': k4, 'k8': k8,
            'n0': n0, 'n1': n1, 'n2': n2, 'chi': chi,
            'm01_total': int(m01_8.sum()), 'm12_total': int(m12_8.sum()),
            'm01': m01_8, 'm12': m12_8}

## 4. Descriptores atribuídos, identidades y exportación

Se calculan descriptores **comparables entre matrices del mismo atlas**: áreas, perímetros de bordes primal, longitudes de 1-generadores, agujeros, multiplicidades, número de contactos por pareja, grados e identidad de Euler. Los controles RAG y geométricos se extraen de las mismas imágenes.

In [ ]:
def compute_connectome_hig(c, make_hig8=True):
    q=labels_from_correlations(c)
    ids, edges, geom, cuts, carriers, holes, m01, m12=construct(q)
    nreg=m12.shape[1]-1
    n0,n1=m01.shape
    assert n0-n1+m12.shape[1]==2, 'Falla identidad Euler HIG4'
    assert len(cuts)==sum(holes.values()), 'Falla correspondencia de agujeros HIG4'
    assert m01[:,len(geom):].nnz==0
    region_area=np.bincount(ids.ravel(),minlength=nreg)
    region_perimeter=np.zeros(nreg,dtype=int)
    pair_len=Counter()
    for va,vb,pair,frame in edges:
        a,b=pair
        if a<nreg: region_perimeter[a]+=1
        if b<nreg: region_perimeter[b]+=1
        if a<nreg and b<nreg: pair_len[pair]+=1
    assert sum(region_perimeter)==2*sum(pair_len.values())+sum(1 for e in edges if e[3])
    gtypes=Counter(g['type'] for g in geom)
    deg=np.asarray(m12[:len(geom),:nreg].astype(bool).sum(axis=0)).ravel()
    # cálculo escalable de grados de vértices de frontera
    counter=Counter(v for a,b,_,_ in edges for v in (a,b))
    junctions=sorted(v for v,k in counter.items() if k>=3)
    h8=_derive_hig8(q, ids, geom, cuts, holes,m01,m12,junctions) if make_hig8 else None
    region_pairs=Counter(g['pair'] for g in geom)
    base={'N':len(c),'regions4':nreg,'regions8':len(h8['regions']) if h8 else np.nan,
          'holes4':len(cuts),'holes8':h8['k8'] if h8 else np.nan,
          'diagonal_occurrences':h8['d'] if h8 else np.nan,
          'diagonal_merges':h8['m'] if h8 else np.nan,
          'v0':n0,'v1':n1,'v2_out':m12.shape[1],'chi4':n0-n1+m12.shape[1],
          'chi8':h8['chi'] if h8 else np.nan,'n_seg':gtypes['seg'],
          'n_enc0':gtypes['enc0'],'n_enc1':gtypes['enc1'],
          'total_frontier':len(edges),'frontier_interior':sum(not e[3] for e in edges),
          'rag_edges':len(pair_len), 'repeated_pairs':sum(k>1 for k in region_pairs.values()),
          'max_pair_contacts':max(region_pairs.values(),default=0),
          'mean_region_area':float(np.mean(region_area)),
          'max_region_area':int(np.max(region_area)),
          'mean_perimeter':float(np.mean(region_perimeter)),
          'max_perimeter':int(np.max(region_perimeter)),
          'mean_1gen_length':float(np.mean([len(g['edges']) for g in geom])) if geom else 0.,
          'max_1gen_length':max([len(g['edges']) for g in geom],default=0),
          'regions_with_holes':sum(h>0 for h in holes.values()),
          'max_holes_region':max(holes.values(),default=0),
          'max_region_incidence_degree':int(max(deg,default=0)),
          'mean_signed_r':float(np.mean(c[~np.eye(len(c),dtype=bool)])),
          'mean_abs_r':float(np.mean(np.abs(c[~np.eye(len(c),dtype=bool)]))) }
    return base, {'q':q,'ids':ids,'edges':edges,'geom':geom,'cuts':cuts,'m01':m01,'m12':m12,
                  'h8':h8,'areas':region_area,'perimeters':region_perimeter}

if paths:
    t0=time.time(); features0, structure0=compute_connectome_hig(first)
    display(pd.Series(features0).to_frame('valor'))
    print('Tiempo primera matriz:',round(time.time()-t0,3),'s')
    fig, ax=plt.subplots(1,2,figsize=(10,4)); ax[0].imshow(first,vmin=-1,vmax=1,cmap='coolwarm');ax[0].set_title('Correlación r');
    ax[1].imshow(structure0['q'],cmap='tab10',vmin=0,vmax=5);ax[1].set_title('Categorías firmadas + diagonal');
    plt.tight_layout();plt.show()

## 5. Análisis por cohorte y comparación científica

Se calculan descriptores para todas las matrices; para ejecutar clasificación es **obligatorio** un `manifest.csv` con etiqueta y sujeto, y al menos dos sujetos por clase. El código usa separación por sujeto (o familia si se proporciona). Cada grupo de matrices de un sujeto se mantiene unido. Todos los pasos de imputación, estandarización y clasificación se ajustan dentro de cada partición de entrenamiento. No interpretar resultados de cohortes pequeñas como prueba de biomarcador clínico.

In [ ]:
manifest_path=OUT/'manifest.csv'
manifest=pd.read_csv(manifest_path) if manifest_path.exists() else None
if manifest is not None:
    if not {'filename','subject','label'}.issubset(manifest.columns): raise ValueError('manifest: columnas filename,subject,label obligatorias')
    manifest['filename']=manifest['filename'].map(lambda x:pathlib.Path(str(x)).name)
    assert manifest.filename.is_unique, 'Un filename no debe repetirse'
    print('Manifest:',len(manifest),'filas')
rows=[]; errors=[]
for p in paths:
    try:
        c=load_matrix(p); v,_=compute_connectome_hig(c)
        rows.append({'filename':p.name,**v})
    except Exception as e:
        errors.append((p.name,str(e)))
results=pd.DataFrame(rows)
if errors: display(pd.DataFrame(errors,columns=['filename','error']))
if len(results):
    if manifest is not None: results=results.merge(manifest,on='filename',how='inner',validate='one_to_one')
    display(results.head())
    results.to_csv(ROOT/'connectome_hig_features.csv',index=False)
    print('CSV de características:',ROOT/'connectome_hig_features.csv')

In [ ]:
# Familias de características: mismas muestras y mismos splits para comparaciones pareadas.
BASELINE=['mean_signed_r','mean_abs_r']
RAG=BASELINE+['regions4','rag_edges','frontier_interior','mean_region_area','max_region_area']
GEOMETRIC=RAG+['mean_perimeter','max_perimeter','mean_1gen_length','max_1gen_length']
HIG4=GEOMETRIC+['holes4','n_seg','n_enc0','n_enc1','repeated_pairs','max_pair_contacts','regions_with_holes','max_holes_region','max_region_incidence_degree']
HIG48=HIG4+['regions8','holes8','diagonal_occurrences','diagonal_merges']
FEATURE_SETS={'matrix_summaries':BASELINE,'RAG':RAG,'RAG_plus_geometry':GEOMETRIC,'HIG4_attributed':HIG4,'HIG4_and_HIG8':HIG48}
if manifest is None: print('Sin etiquetas: extracción descriptiva solamente; crea manifest.csv para clasificación.')
elif len(results):
    y=results.label.astype(str).values
    groups=results['family'].fillna(results['subject']).astype(str).values if 'family' in results.columns else results.subject.astype(str).values
    ngrp=len(set(groups)); classes=sorted(set(y))
    class_grp={c:len(set(groups[y==c])) for c in classes}
    if len(classes)!=2 or min(class_grp.values())<2: print('No se ejecuta CV: se necesitan dos clases y al menos dos grupos por clase.',class_grp)
    else:
        k=min(5,min(class_grp.values()))
        cv=StratifiedGroupKFold(n_splits=k,shuffle=True,random_state=42)
        scores=[]; X_all=results
        for fold,(tr,te) in enumerate(cv.split(X_all,y,groups)):
            if len(set(y[tr]))!=2 or len(set(y[te]))!=2: continue
            for name,columns in FEATURE_SETS.items():
                model=make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),LogisticRegression(max_iter=2500,class_weight='balanced'))
                model.fit(X_all.iloc[tr][columns],y[tr]); pred=model.predict(X_all.iloc[te][columns]); prob=model.predict_proba(X_all.iloc[te][columns])[:,list(model.classes_).index(classes[1])]
                scores.append({'fold':fold,'features':name,'balanced_accuracy':balanced_accuracy_score(y[te],pred),
                               'roc_auc':roc_auc_score((y[te]==classes[1]).astype(int),prob),
                               'n_test':len(te),'groups_test':len(set(groups[te]))})
        validation=pd.DataFrame(scores)
        if len(validation):
            display(validation.groupby('features')[['balanced_accuracy','roc_auc']].agg(['mean','std']))
            validation.to_csv(ROOT/'connectome_cv_scores.csv',index=False)
        else: print('Ninguna partición contenía las dos clases: no hay AUC válida.')

## 6. Exportación reproducible y controles

Para el artículo debe informarse del número real de sujetos, su distribución por clases, atlas/orden ROI, si las matrices eran `r` o Fisher `z`, tratamiento de la diagonal, umbrales, QC y protocolo de validación. No presentamos AUC cuando faltan etiquetas o hay pocos sujetos. Los datos sintéticos solo prueban que el programa funciona.

In [ ]:
metadata={'atlas_roi_count':EXPECTED_N,'input_scale':INPUT_SCALE,'thresholds':list(THRESHOLDS),'diagonal_mode':DIAGONAL_MODE,
          'nb_matrices':len(results),'files':results.filename.tolist() if len(results) else [],
          'origin':'Adaptation of HIGM48_editor_20x20_Colab.ipynb; intrínseco HIG4 core and comb HIG8 quotient'}
(ROOT/'experiment_config.json').write_text(json.dumps(metadata,indent=2,ensure_ascii=False))
if paths:
    # Export matrices de incidencia en formato disperso y atributos de la primera muestra como ejemplo
    _, structure0=compute_connectome_hig(load_matrix(paths[0]))
    sparse.save_npz(ROOT/'HIG4_M01_example.npz',structure0['m01']); sparse.save_npz(ROOT/'HIG4_M12_example.npz',structure0['m12'])
    if structure0['h8']:
        sparse.save_npz(ROOT/'HIG8_M01_example.npz',structure0['h8']['m01']); sparse.save_npz(ROOT/'HIG8_M12_example.npz',structure0['h8']['m12'])
    np.save(ROOT/'HIG4_region_ids_example.npy',structure0['ids']); np.save(ROOT/'HIG4_labels_example.npy',structure0['q'])
    np.savez_compressed(ROOT/'HIG4_region_geometry_example.npz',area=structure0['areas'],perimeter=structure0['perimeters'])
zip_path=ROOT/'results_hig_connectomes.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in ROOT.iterdir():
        if p.is_file() and p!=zip_path: z.write(p,p.name)
print('Resultados:',zip_path)
if IN_COLAB: files.download(str(zip_path))

## 7. Prueba unitaria sin datos clínicos (opcional)
Este bloque usa una matriz de correlación **válida por construcción** y una variante con orden fijo. Sirve para verificar el software, **no** para informar de clasificación fMRI.

In [ ]:
rng=np.random.default_rng(7)
X=rng.standard_normal((160,12)); C=np.corrcoef(X,rowvar=False); np.fill_diagonal(C,1.)
f,h=compute_connectome_hig(C)
assert f['chi4']==2 and f['chi8']==2 and f['holes8']-f['holes4']==f['diagonal_occurrences']-f['diagonal_merges']
print('Prueba HIG₄/HIG₈ superada:',{k:f[k] for k in ('regions4','regions8','holes4','holes8','chi4','chi8')})